# Lesson 36 · Window functions — learn to be gentle before you analyze: taming spectral leakage, Hann vs Hamming vs Blackman

**Today's goal**: first understand where spectral leakage comes from, then learn to read the edge differences between Rectangular / Hann / Hamming / Blackman off a comparison table.

> **Main path**: use **Hann** to explain "why be gentle" thoroughly → then scan the table to compare the other three windows (no need to derive a formula for each one). The next stop, the STFT (L43), multiplies every frame by a window.

**Key points**: where the rectangular window's sidelobe leakage comes from; how smooth windows suppress it.

**Why it matters for Aurora**: every STFT frame uses Hann by default; `windows.py` corresponds to this lesson. Pick the wrong window and leakage will pollute the Mel features.

← **Previous lesson** · [L35 · Euler meets the FFT](L35_euler_fft.ipynb)

> Last lesson covered **Euler meets the FFT**: what e^{-2πikn/N} is, and visualizing twiddle factors.
> This lesson explores **window functions**.

## This lesson's story: why do recordings "fade in and fade out"?

When a mixing engineer edits a recording, they often put a "fade in / fade out" at both ends of a clip — the volume rises slowly from 0 to full, then sinks slowly back to 0.

This isn't just taste; it's a **mathematical necessity**: if you cut a stretch of signal out directly, the amplitude at the edges jumps suddenly from some value to 0 — the same as multiplying by a rectangular function. The FFT doesn't know "this is a cut boundary"; it treats the jump as a real high-frequency component and produces **sidelobe pollution** in the spectrum.

### Why does a hard jump at the boundary create high frequencies?

Imagine a string on an instrument: you're playing a pure 440 Hz A. Now suppose you suddenly clamp the string hard with your finger — what happens? Not silence, but a harsh noise full of high-frequency components. Mathematically this is **spectral leakage**: a discontinuous boundary in the time domain corresponds to high-frequency pollution in the frequency domain. Concretely:

- **Rectangular truncation** = signal × a rectangular function (value 1 inside, 0 outside)
- In the frequency domain, this is equivalent to convolving the original spectrum with a **sinc function**
- The sinc's first sidelobe sits at about -13 dB (about 1/4.5 of the main peak), causing "sidelobe leakage"

**What is "convolution", anyway? (three sentences to build intuition, no strict definition)** Think of convolution as taking a "blur brush" and smearing it over another function: place the sinc shape (one big bump in the middle, a train of ever-smaller ripples on each side) on every narrow, sharp spectral line, so that each line "bleeds" into the shape of a sinc — what was one clean spike still has its center, but now has a train of shrinking "afterglow" on both sides, and that afterglow is the sidelobes. The Fourier transform of a rectangular function comes out exactly sinc-shaped because of a duality between time and frequency: **the "squarer and harder" something is in time (the more abrupt its edges), the more "spread out" it is in frequency (the more sidelobes)**. The full algebraic derivation (why the Fourier transform of a rectangle is exactly a sinc) comes in **L37, deriving the DFT by hand**, and in L38–L39 on the FFT — for now you only need to remember the correspondence: **multiplying by a rectangular window in time ⇔ convolving with a sinc in frequency ⇔ sidelobe leakage**.

**What is a decibel (dB)?** It's a logarithmic scale that measures an amplitude ratio:
$$\text{dB} = 20 \log_{10}(\text{amplitude ratio})$$
- A ratio of 1 gives 0 dB
- A ratio of 10 gives 20 dB
- A ratio of 0.1 gives -20 dB

In the table, "-13 dB" means "the sidelobe amplitude is 1/4.5 of the main peak"; "-58 dB" means "the sidelobe is only about 1/1000 of the main peak". The more negative the dB, the more thoroughly the sidelobes are pushed down.

A **window function** solves this problem: a set of weights as long as the signal, near 0 at both ends and near 1 in the middle. Multiply the signal by these weights point by point, the edges get "gently pressed down", and the FFT sees a smooth transition instead of a sudden jump — greatly reducing sidelobe pollution.

Four common windows:

| Window | Core of the formula | Sidelobe suppression | Main-lobe width |
|---|---|---|---|
| **Rectangular** | w[k]=1 | poor (-13 dB) | narrowest |
| **Hann** | 0.5·(1-cos(2πk/N)) | good (-31 dB) | medium |
| **Hamming** | 0.54-0.46·cos(2πk/N) | better (-41 dB) | medium |
| **Blackman** | 0.42-0.5·cos(2πk/N)+0.08·cos(4πk/N) | very good (-58 dB) | widest |

> Note: the table uses N in the denominator (the **periodic** convention, Aurora/STFT default). NumPy's `np.hanning`/`np.hamming` use the **symmetric** convention (denominator N−1), and the hand-calculation table and whiteboard challenge later in this lesson use exactly that symmetric version — the difference between the two conventions is explained below in the section "Periodic vs Symmetric".

This section's task: implement `describe_window(name, w)`, which reads a window array's first and last values and the position of its peak, to verify the shape differences between the three windows.

## 1. First, look at the window function itself

A window function is a set of weights as long as the signal. Multiply the signal by these weights point by point, and the edges of the clip get pressed down while more of the middle is kept.

- Rectangular: every position is 1 — the edges are left untouched.
- Hann / Blackman: near 0 at both ends — the softest edges.
- Hamming: not 0 at the ends — keeps a little edge energy.

Four numbers are enough to describe a window's boundary behavior: the first value `w[0]`, the last value `w[-1]`, the peak `w.max()` and the peak position `argmax(w)`. Hann / Blackman are near 0 at both ends, pressing the edges almost to zero; Hamming's ends are about 0.08, keeping a little edge energy; Rectangular's ends are 1, no suppression at all. The closer the boundary values are to 0, the lower the spectral sidelobes, but the wider the main lobe — an unavoidable trade-off between sidelobe suppression and spectral resolution.

### Why is Hamming "stronger" than Hann?

Here's something counterintuitive: Hann's ends are 0 (the edges fully suppressed), while Hamming's ends are about 0.08 (some edge energy kept). Yet in the table Hamming's sidelobes (-41 dB) are lower than Hann's (-31 dB)!

The reason lies in **how the sinc sidelobes are built up in the frequency domain**. Hann's "full suppression" (zero at both ends) has some side effects in the frequency domain: there is still a slight discontinuity at the edges (jumping from a nonzero value to 0). Hamming's "keep a little energy" design (0.08 at the ends) is carefully calculated so that this 0.08 produces **destructive interference** with the spectrum of the middle part — sidelobe components at different frequencies cancel each other. It's like the principle of noise-cancelling headphones: add an inverted waveform to cancel the sound you don't want.

**Where 0.54 and 0.46 come from**: they weren't picked at random. They come from numerical optimization: researchers searched by computer for the best pair of constants that makes the frequency-domain sidelobes lowest. The answer was 0.54 and 0.46 (which add up to exactly 1). Hann's corresponding coefficients are 0.5 and 0.5, a special case.

**The root of the sidelobe/main-lobe trade-off**: this isn't a flaw in our design; it's a **fundamental limit** of Fourier analysis. Between time and frequency, narrow and wide are relative — you can't make a signal "narrow" in both domains at once. It's like Heisenberg's uncertainty principle: position and momentum can't both be pinned down exactly. For window functions:

- The **harder you squeeze in time** (edges closer to 0) → the **more complex the shape in frequency** (the main lobe widens, and the sidelobes spread out more)
- You can only choose what you care about more: suppressing sidelobes first (Hamming / Blackman), or keeping the main lobe narrow first (Rectangular)?

The STFT's use of the Hann window is a compromise: medium sidelobe suppression (-31 dB, good enough), medium main-lobe width, and simple to compute.

## Into the lab: turning sound into arrays you can inspect

Here we use a short sine of 64 samples at a sample rate (sr) of 200 Hz, so the result of `signal * window` can be printed directly, making it easy to confirm whether the edge values have been pressed down.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
from aurora.audio import hann, hamming, blackman
print('Ready')

## Hands-on: how a sequence becomes a signal, step by step

Change `sample_rate` or `duration` and watch how the number of samples `N = int(sample_rate * duration)` changes; the window-weight array in the next cell will have the same length `N`.

In [ ]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('sample index n =', n)
print('time axis t =', np.round(t, 3))
print('angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


## Code experiment: comparing the edges of several window functions

Print `w[:3]` and `w[-3:]` and see the difference directly: the rectangular window's edge values are 1.0, while the Hann window's edges are close to 0.

### Periodic vs Symmetric Hann: how Aurora and NumPy differ

The same Hann formula comes in two common conventions:

| | `np.hanning(N)` (symmetric) | `aurora.hann(N)` (periodic / DFT-even) |
|---|---|---|
| Formula | `w[n] = 0.5(1 - cos(2πn/(N-1)))` | `w[n] = 0.5(1 - cos(2πn/N))` |
| First value `w[0]` | 0 | 0 |
| Last value `w[-1]` | ≈ 0 | ≈ 0.095 (N=10) / 0.0024 (N=64) |
| Peak position | `N//2 - 1` = 4 (N=10) | `N//2` = 5 (N=10) |
| Typical use | FIR filter design | seamless STFT frame stitching (Aurora default) |

**Why are there two formulas?**

Picture it simply: when you split a signal into overlapping frames to do an STFT, neighboring frames are added together in the overlap region. If the window's last value `w[-1]` is exactly 0 (symmetric), the two frames leave a "fault line" where they're added — energy is lost at the boundary. But if the last value isn't 0 (periodic), the weighted sum of neighboring frames in the overlap stays constant, giving a "seamless join". This matters for the accuracy of the STFT's frequency analysis.

**A concrete example**: the two formulas compared for N=10
```
Symmetric (NumPy): k=0,1,2,...,9 → denominator = N-1 = 9
  w[0] = 0.5(1-cos(0))       = 0
  w[1] = 0.5(1-cos(2π/9))    ≈ 0.1170
  w[4] = 0.5(1-cos(8π/9))    ≈ 0.9698 (peak)
  w[9] = 0.5(1-cos(18π/9))   = 0

Periodic (Aurora): k=0,1,2,...,9 → denominator = N = 10
  w[0] = 0.5(1-cos(0))       = 0
  w[1] = 0.5(1-cos(2π/10))   ≈ 0.0955
  w[5] = 0.5(1-cos(10π/10))  = 1.0 (peak in the middle)
  w[9] = 0.5(1-cos(18π/10))  ≈ 0.0955 (not 0!)
```

**The key difference**: the symmetric w[9]=0 makes the window "cut off abruptly", like the rectangular truncation problem from before; the periodic w[9]≈0.095 lets the window "step down gracefully", joining smoothly with the next frame. That's why the STFT uses the periodic version.

This lesson's code experiments (cells 11, 21 and 31) use `np.hanning`/`np.hamming` to show the intuitive shape of symmetric windows; the Aurora source (`aurora.audio.windows`) and the exercise (cell 14) use the `periodic=True` version. The two sets of numbers differ, but the physical meaning is the same — both press down the edge energy.

In [ ]:
import numpy as np

N = 16
windows = {
    'rectangular': np.ones(N),
    'hann': np.hanning(N),
    'hamming': np.hamming(N),
}

for name, w in windows.items():
    print(name)
    print('  first 5 =', np.round(w[:5], 3))
    print('  last  5 =', np.round(w[-5:], 3))


## 2. ✏️ Implement `describe_window(name, w)`

**Work out the key points of a Hann window (N=8) by hand**:

Formula (**symmetric** convention, denominator N−1, same as `np.hanning`; the table in the introduction is the periodic version, denominator N): `w[k] = 0.5 · (1 - cos(2π·k / (N-1)))`, k = 0, 1, ..., N-1

| k | angle 2πk/7 | cos(...) | w[k] |
|---|---|---|---|
| 0 | 0 | 1 | **0** (left end = 0) |
| 1 | 2π/7 | 0.6235 | 0.1883 |
| 3 | 6π/7 | -0.9009 | **0.9505** (near the peak) |
| 7 | 2π | 1 | **0** (right end = 0) |

**Patterns**:
- Symmetric window: `w[0] ≈ w[-1] ≈ 0`, peak near `N//2 - 1`
- Hamming: ends are not 0 (≈ 0.08), keeping a little edge energy
- Rectangular: `w[0] = w[-1] = 1.0` (no fade)

`describe_window` should return a 4-tuple `(first, last, peak, peak_idx)`, in this order:
- `first`: `w[0]`
- `last`: `w[-1]`
- `peak`: `w.max()`
- `peak_idx`: `int(np.argmax(w))` ← what is `argmax`? It returns **the index of the position where the maximum sits**. For example, `np.argmax([0.5, 0.9, 0.3])` returns 1 (because the maximum, 0.9, is at position 1)

### Before writing code, fill in the variable table

Be clear on three things before writing `describe_window`:
- Input: `w` (an array of window weights of shape `(N,)`, values in 0–1)
- Key steps (remember these four):
  1. `w[0]` — the first value (rectangular 1, Hann/Blackman near 0, Hamming ≈ 0.08)
  2. `w[-1]` — the last value (rectangular 1, Hann last = 0, Blackman near 0, Hamming ≈ 0.08; but periodic Hann/Hamming may not end at 0)
  3. `w.max()` — the peak (every window's peak is close to or equal to 1)
  4. `int(np.argmax(w))` — the index where the peak sits. `argmax` returns the position of the maximum: `np.argmax([0.1, 0.9, 0.5])` → 1
- Return: a 4-tuple `(first, last, peak, peak_idx)` — first value, last value, peak, and peak index, in that order

In [ ]:
def describe_window(name, w):
    """
    Describe the key properties of a window function.
    
    Inputs:
      name: the window's name (a string), used for printing
      w: array of window weights, shape (N,), values in [0, 1]
    
    Returns: a 4-tuple (first, last, peak, peak_idx)
      first: w[0], the first value
      last: w[-1], the last value
      peak: w.max(), the peak
      peak_idx: position of the peak (an integer index)
    
    Steps:
      1. read w[0], w[-1] and w.max() from w
      2. find the peak position with np.argmax(w); remember to convert it to int
      3. print these four values (handy for debugging)
      4. return the 4-tuple
    """
    # ✏️ TODO: replace the lines below and implement the steps above
    print(name, 'window array length', len(w))
    raise NotImplementedError("TODO: extract first/last/peak/peak position and return the 4-tuple (first, last, peak, peak_idx)")

## 3. Call it on three windows + automatic check

In [ ]:
N = 64
windows = {'Hann': hann(N), 'Hamming': hamming(N), 'Blackman': blackman(N)}
all_ok = True
for name, w in windows.items():
    assert w.shape[0] == N
    try:
        result = describe_window(name, w)
    except (NotImplementedError, TypeError):
        print('⬜ not implemented yet')
        all_ok = False
        break
    if result is None:
        print('⬜ not implemented yet')
        all_ok = False
        break
    first, last, peak, peak_idx = result
    assert abs(first - w[0]) < 1e-9, f'{name}: wrong first value'
    assert abs(last - w[-1]) < 1e-9, f'{name}: wrong last value'
    assert abs(peak - w.max()) < 1e-9, f'{name}: wrong peak value'
    assert peak_idx == int(np.argmax(w)), f'{name}: wrong peak_idx'

if all_ok:
    assert abs(hann(N)[0]) < 1e-9, 'Hann window should start at 0 (periodic DFT-even window; the last value w[-1] ≈ 0.0024 for N=64, not 0)'
    print('\n✅ Window-function properties checked.')

## 4. Plot the three windows side by side

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3.5))
for name, w in windows.items():
    plt.plot(w, label=name)
plt.title(f'Window functions (N={N})')
plt.xlabel('sample'); plt.ylabel('weight')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4.1 Before and after windowing: same signal, different edges

Below we keep the same sine and change only the window. First look at the edges in the time domain, then at the energy next to the main peak in the frequency domain.


## 4.0 What windowing does to STFT framing

Earlier we said the Aurora STFT multiplies every frame by a Hann window. How exactly?

Imagine splitting a long audio clip into short overlapping frames, 256 samples per frame, with neighboring frames overlapping by 128 points:

```
Original signal: [===frame1===|====frame2====|====frame3====| ...]
Frame length:     |--------|
Overlap:            |------| (128 samples)
```

Window each frame:

```
frame1 → multiply by Hann window w → xw1 = frame1 * w
frame2 → multiply by Hann window w → xw2 = frame2 * w
...
```

**Energy conservation** (Constant Overlap-Add, COLA):
With periodic Hann (not symmetric), when the overlapping parts of neighboring frames are added together, the total stays constant. In other words:
- the last 128 points of xw1 + the first 128 points of xw2 = a constant (no attenuation or boost from the windowing)

This is called a "seamless join". Conversely, with symmetric Hann (last value 0), the overlap has an "energy gap", and the whole signal comes out weakened after STFT→ISTFT — that's why the STFT uses the periodic version.

**A numerical check** follows below: using two windows and two overlapping frames, compare the energy before and after windowing.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

N = 64
n = np.arange(N)
signal = np.sin(2*np.pi*5.5*n/N)  # non-integer number of periods, so the edges won't line up
windows = {
    'rectangular': np.ones(N),
    'hann': np.hanning(N),
    'hamming': np.hamming(N),
    'blackman': np.blackman(N),  # added to match the heading
}

plt.figure(figsize=(9, 3.2))
for name, w in windows.items():
    plt.plot(signal * w, label=name)
plt.title('same signal × different windows')
plt.legend(); plt.grid(True, alpha=.25); plt.show()

plt.figure(figsize=(9, 3.2))
for name, w in windows.items():
    spectrum = np.abs(np.fft.rfft(signal * w))
    plt.plot(spectrum / spectrum.max(), label=name)
plt.title('normalized spectrum after windowing')
plt.legend(); plt.grid(True, alpha=.25); plt.show()


### Interlude: how exactly does a jump "need" high-frequency sine waves?

In the code above, `signal = np.sin(2*np.pi*5.5*n/N)` deliberately picks a **non-integer** number of periods (5.5 periods, not 5 or 6). This is the key: the FFT treats these 64 points in the window as "one loop played over and over forever" — if you snip this little stretch out, join its end to its start and play it on repeat, then where the 5.5 periods finish, the waveform's height doesn't match the start of the next round, and you get a "step" (a jump). An integer number of periods (say 6.0) joins up end to start exactly, with no step.

So what does that step mean in the spectrum? Think of it another way: suppose all you have is a pile of pure sine waves with "whole numbers of periods" (1 period, 2 periods, 3 periods…), and you want to add them up into a waveform with a step in it. How many sine waves would you need? The answer: a great many, and **the higher-frequency ones are precisely the ones responsible for the narrow, steep turn near the step** — the sharper the turn, the more it relies on higher and higher frequencies to "trace the outline". That's the root reason rectangular truncation (no window) creates lots of high-frequency sidelobes in the spectrum: an edge discontinuity "inherently needs" high-frequency sine waves to piece it together.

To make this easier to see, below we switch to a more extreme example that's also easier to draw — an ideal square wave (jumping suddenly from -1 to +1, a more dramatic jump than the truncated sine above) — and look directly at what "drawing a jump by adding up sine waves" looks like.

In [ ]:
# Demo: how many sine waves must be added up to "draw" a jump discontinuity?
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 2*np.pi, 1000, endpoint=False)
square = np.sign(np.sin(x))  # ideal square wave: jumps suddenly at 0 and pi, an extreme example of a "jump"

plt.figure(figsize=(9, 3.5))
plt.plot(x, square, 'k--', label='ideal square wave (a true jump)', linewidth=1.5)
for n_harmonics in [1, 3, 7, 21]:
    approx = np.zeros_like(x)
    for k in range(1, n_harmonics + 1, 2):  # the square wave's Fourier series needs only odd harmonics 1,3,5,...
        approx += (4 / (np.pi * k)) * np.sin(k * x)
    plt.plot(x, approx, label=f'sum of {(n_harmonics + 1) // 2} sine harmonics')
plt.title('To "draw" a jump you need more and more, ever higher-frequency sine waves')
plt.legend(fontsize=8); plt.grid(True, alpha=.25); plt.show()

print('With just 1 sine wave: you can only draw a smooth sine curve, nothing like a right-angle jump.')
print('Adding up to the 21st harmonic: the jump is much steeper, but the edge still has a little "ripple/overshoot" —')
print('this is called the Gibbs phenomenon: even with infinitely many sine waves the ripple never disappears, it only gets squeezed narrower.')
print('This is exactly where the rectangular window\'s sidelobes come from: a jump inherently needs infinitely many high-frequency sines to be rebuilt exactly;')
print('the FFT can only use a finite number of bins, and the high-frequency energy it can neither fully assemble nor cleanly cancel becomes the sidelobes in the spectrum.')
print('When L37 derives the DFT by hand, it gives the rigorous coefficient derivation behind this (the Fourier series coefficient formula).')


## 5. Source reading: mapping the concepts back to Aurora

After finishing the window-function experiments, open these two files:

- `src/aurora/audio/io.py`: `sine` / `chirp` / `read_wav` / `write_wav`
- `src/aurora/audio/windows.py`: `hann` / `hamming` / `blackman`

While reading, compare only three things: the function inputs, the returned arrays, and the array lengths.


## 🎨 Figure: from window → STFT framing → spectrum → mel

Take a first look at what L37–L47 will do, to get a unified picture.

In [ ]:
from aurora.audio import sine
import aviz as aviz; aviz.style()
long = sine(220.0, duration=0.05, sample_rate=16000)
aviz.windows_overlay(64);

In [ ]:
aviz.framing(long, frame_len=256, hop=128, n_frames=5);   # STFT framing

In [ ]:
aviz.spectrogram(long);                                   # power spectrogram

In [ ]:
aviz.mel_filterbank_plot(20, 256, 16000);                 # mel filterbank = a matrix

In [ ]:
aviz.mel_spectrogram_plot(long, 16000, 40);               # mel spectrogram

In [ ]:
N = 16
signal = np.ones(N)
for window_name, window in [('rect', np.ones(N)), ('hann', np.hanning(N)), ('hamming', np.hamming(N)), ('blackman', np.blackman(N))]:
    shaped = signal * window
    edge_energy = shaped[0]**2 + shaped[-1]**2
    print(f'{window_name:7s} | edge values=({shaped[0]:.3f}, {shaped[-1]:.3f}) | edge energy={edge_energy:.3f}')


In [ ]:
# Demo: checking the COLA condition (energy conservation)
# With periodic Hann windowing and overlap = 50% (hop_size = N//2), the windowed frames should add up to a constant

N = 64
from aurora.audio import hann as aurora_hann
from numpy import hanning

# Aurora's periodic Hann (default parameter periodic=True)
w_periodic = aurora_hann(N)
# NumPy's symmetric Hann (the default)
w_symmetric = hanning(N)

# simulate two overlapping frames, overlap = 50%
hop = N // 2  # 50% overlap
frame1 = np.ones(N)  # simplification: all ones
frame2 = np.ones(N)

# windowed frames
xw1_periodic = frame1 * w_periodic
xw2_periodic = frame2 * w_periodic

xw1_symmetric = frame1 * w_symmetric
xw2_symmetric = frame2 * w_symmetric

# add in the overlap region (last hop points of frame1 + first hop points of frame2)
overlap_sum_periodic = xw1_periodic[hop:] + xw2_periodic[:hop]
overlap_sum_symmetric = xw1_symmetric[hop:] + xw2_symmetric[:hop]

print("Periodic Hann (Aurora):")
print(f"  first 3 overlap values: {np.round(overlap_sum_periodic[:3], 4)}")
print(f"  last 3 overlap values: {np.round(overlap_sum_periodic[-3:], 4)}")
print(f"  close to constant? {np.allclose(overlap_sum_periodic, overlap_sum_periodic[0], atol=0.01)}")

print("\nSymmetric Hann (NumPy):")
print(f"  first 3 overlap values: {np.round(overlap_sum_symmetric[:3], 4)}")
print(f"  last 3 overlap values: {np.round(overlap_sum_symmetric[-3:], 4)}")
print(f"  close to constant? {np.allclose(overlap_sum_symmetric, overlap_sum_symmetric[0], atol=0.01)}")
print(f"  (note: symmetric Hann ends at 0, so the second half of the overlap has a gap)")

print("\n💡 That's why the STFT must use the periodic version: energy is conserved → the frequency analysis stays accurate.")

## Parameter experiment: comparing sidelobe heights directly

Take the same sine (with a frequency falling exactly between two FFT bins), window it with Rectangular and with Hann, take the FFT, and observe that Rectangular's spectral sidelobes sit only about 13 dB below the main lobe, while Hann's are about 31 dB down. The gap comes from edge handling: Rectangular's hard cut manufactures fake high frequencies, while Hann's smooth transition suppresses the sidelobes heavily. That's why the STFT almost always uses a Hann window.

Going further: change the `5.5` in `signal = np.sin(2*np.pi*5.5*n/N)` to an integer (such as `6.0`) and the rectangular window's sidelobes disappear — when the frequency is a whole multiple, the signal is exactly continuous across the FFT's periodic boundary, the edge jump is zero, and the sidelobe pollution vanishes with it.

## Appendix A · Before L37: a 5-minute preview of divide-and-conquer thinking (not a new lesson)

You've finished the windowing step:

```text
raw clip x  ×  window weights w  =  windowed clip xw
```

The next lesson, L37, asks a new question:

> Which frequencies are actually mixed into `xw`?

The DFT's brute-force approach is direct: line up every candidate frequency against `xw` and compare,
computing "how strong this frequency component is". The formula reads:

```text
X[k] = Σ x[n] · e^{-2πikn/N}
```

You don't need to derive it now; just remember three things:

1. L36's output is an array: `xw = x * window`
2. L37 computes all the frequency components by brute force with a double loop
3. Only L38–L39 ask: can we avoid recomputing every frequency from scratch?

The FFT's divide-and-conquer intuition is like "finding a floor in a tall tower by halving":
if you check every floor one by one, it costs a lot; if you can keep splitting the problem into two halves,
you skip a large amount of repeated work.

In the FFT, "splitting into two halves" isn't arbitrary; the split is into:

- even-index samples: `x[0], x[2], x[4], ...`
- odd-index samples: `x[1], x[3], x[5], ...`

L37 still only writes the brute-force DFT; L38 draws the butterfly diagram; L39 writes the recursive FFT.
For this lesson you only need to fix the entry point firmly in mind:

```text
signal → window → xw → DFT/FFT
```

If you get stuck, look back at:

- L06 / L35: the twiddle factor `e^{-2πikn/N}`
- L07: a complicated signal can be seen as a sum of sines
- L21: the DFT can be written as a matrix multiplication `X = W @ x`


## Optional warm-up · use `np.fft.fft` to see L37's input and output

The cell below isn't meant to teach you the FFT early; it only looks at shapes:

- Input: a real array `xw` of length `N`
- Output: a complex array `X`, still of length `N`
- L37 will hand-write the brute-force version of this transform


In [ ]:
import numpy as np

N = 8
n = np.arange(N)

# one full period of a sine wave; L37 will break it into frequency components
x = np.sin(2 * np.pi * 1 * n / N)

# symmetric Hann here is just for an intuitive demo; the Aurora STFT defaults to periodic Hann
w = np.hanning(N)
xw = x * w

X = np.fft.fft(xw)

print("x  shape:", x.shape, "first 4:", np.round(x[:4], 3))
print("w  shape:", w.shape, "first 4:", np.round(w[:4], 3))
print("xw shape:", xw.shape, "first 4:", np.round(xw[:4], 3))
print("X  shape:", X.shape, "first 4:", np.round(X[:4], 3))

assert x.shape == w.shape == xw.shape == X.shape == (N,)
assert np.iscomplexobj(X), "FFT output should be complex: magnitude + phase"

print("✅ Warm-up done: L37 will explain how X[k] is computed")


## Wrap-up

You can now use `hann(N)`, `hamming(N)` and `blackman(N)` to generate window weights, and treat `signal * w` as putting "soft padding" on the edges. Printing the first and last values shows the pressing-down effect directly. This corresponds to the first step of the Aurora STFT pipeline: the three functions in `aurora.audio.windows` are called frame by frame when `stft` splits the signal into frames. The next section, L37 (deriving the DFT), heads into the FFT implementation, and today's window weights will be fed straight into the discrete Fourier transform as `xw = x * w`.

From the next lesson on, the question switches from "how do we smooth the edges" to:

> Which frequencies are in this windowed clip `xw`?

L37 first hand-writes the DFT with a double loop;
L38 draws the repeated computation as a butterfly diagram;
L39 writes this divide-and-conquer idea as a recursive FFT.

If you can explain clearly:

```text
signal → window → xw → DFT/FFT
```

you're ready for the next lesson.



In [ ]:
# Quick check: start from a short sequence and confirm each step's output
import numpy as np

sample_rate = 8
duration = 1.0
freq = 1.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
x = np.sin(angle)

print('1) what should N be?', N)
print('2) n is the sample index:', n)
print('3) t is each point\'s time in seconds:', np.round(t, 3))
print('4) angle is each point\'s angle on the circle:', np.round(angle, 3))
print('5) x is the final waveform:', np.round(x, 3))


---
⬇️ **Final check**: you've read the wrap-up; now complete the whiteboard challenge below before ticking the self-review.


## ✏️ Whiteboard challenge: window functions by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: For the Hann window (symmetric convention) `w[k] = 0.5·(1-cos(2πk/(N-1)))` with N=4:
- w[0] = ?
- w[1] = ? (cos(2π/3) = -0.5)
- w[3] = ? (symmetric with w[0])

**Question 2**: The first values (w[0]) of the rectangular window and the Hann window are ? and ?, respectively

**Question 3**: For symmetric Hann (denominator N-1), when N is even, at which index is the peak?
- Steps by hand: the peak occurs where cos(2πk/(N-1)) = -1, and then w[k] = 0.5·(1-(-1)) = 1.0
- For cos to equal -1, the angle must be an odd multiple of π: π, 3π, 5π...
- When 2πk/(N-1) = π, k = (N-1)/2
- If N=8, then k = 3.5 (not an integer, so the peak is near k=3 or k=4)
- If N=10, then k = 4.5 (near k=4, which is N//2-1)

**Question 4**: Why do window functions reduce spectral leakage?
(Hint: rectangular truncation = multiplying by a rectangular function → the FFT gets sinc sidelobes; Hann's gradual taper → much smaller sidelobes)

**Question 5**: Which suppresses sidelobes more strongly, Hann or Blackman? What's the cost? (main-lobe width)

**Question 6**: How does the difference in the last value (w[-1]) between symmetric Hann and periodic Hann affect STFT framing?

When you've finished the derivation, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: Hann(N=4)
N4 = 4
w4 = np.array([0.5*(1 - np.cos(2*np.pi*k/(N4-1))) for k in range(N4)])
assert np.isclose(w4[0], 0.0, atol=1e-12), f"w[0] should be 0, got {w4[0]}"
assert np.isclose(w4[1], 0.5*(1 - np.cos(2*np.pi/3)), atol=1e-12)
assert np.isclose(w4[3], 0.0, atol=1e-12), f"w[3] should be 0, got {w4[3]}"
print(f"Q1 ✅  Hann(N=4): {np.round(w4, 4)}")
print(f"      w[0]={w4[0]:.4f}, w[1]={w4[1]:.4f}, w[3]={w4[3]:.4f}")

# Q2: rectangular starts at 1, Hann starts at 0
rect_first = 1.0
hann_first = 0.0
assert rect_first == 1.0 and np.isclose(hann_first, 0.0)
print(f"Q2 ✅  rectangular window w[0]={rect_first}, Hann window w[0]={hann_first} (Hann presses both ends to 0)")

# Q3: Hann peak position (periodic for N=64; N//2-1 for symmetric)
from aurora.audio import hann
w64 = hann(64)
peak_idx = int(np.argmax(w64))
# periodic Hann with N=64 peaks at N//2 = 32
# symmetric Hann with N=64 peaks at N//2-1 = 31
print(f"Q3 ✅  periodic Hann(N=64) peaks at idx={peak_idx} (should be 32)")
print(f"      for symmetric Hann(N=10), the peak is at N//2-1 = {10//2-1} (k=4)")

# Q4: spectral leakage — rectangular sidelobes vs Hann (numerical check)
from aurora.audio import sine
N = 256
sig = sine(5.5, duration=N/1000, sample_rate=1000)  # non-integer frequency
rect_spec = np.abs(np.fft.rfft(sig * np.ones(N)))
try:
    hann_w = np.hanning(N) if N == 64 else hann(N)
except:
    hann_w = np.hanning(N)
hann_spec = np.abs(np.fft.rfft(sig * hann_w))
# sidelobe: the maximum outside the main lobe
main_bin = int(np.argmax(rect_spec))
sidelobe_rect = np.max(np.delete(rect_spec, range(max(0,main_bin-2), min(len(rect_spec),main_bin+3))))
sidelobe_hann = np.max(np.delete(hann_spec, range(max(0,main_bin-2), min(len(hann_spec),main_bin+3))))
print(f"Q4 ✅  rectangular sidelobe peak={sidelobe_rect:.3f}, Hann sidelobe peak={sidelobe_hann:.3f}")
print(f"      Hann sidelobes are {20*np.log10(max(sidelobe_rect, 1e-6)/max(sidelobe_hann, 1e-6)):.1f} dB lower than rectangular")

# Q5: Hann vs Blackman sidelobes and main lobe
print(f"Q5 ✅  Blackman suppresses sidelobes most strongly (-58 dB vs -31 dB), at the cost of the widest main lobe")
print(f"      Hann is the compromise: medium sidelobes, medium main lobe, simple to compute")

# Q6: how symmetric vs periodic Hann affects STFT framing
print(f"Q6 ✅  Symmetric Hann ends at 0 (energy gap) → energy is lost where STFT frames overlap")
print(f"      Periodic Hann does not end at 0 → energy is conserved (COLA condition), a seamless join")
print(f"      That's why Aurora uses periodic rather than symmetric")

print("\n🎉 Window-function whiteboard challenge passed! Windowing = signal × weights, reducing spectral pollution from cut edges; the STFT uses the periodic version to conserve energy.")

In [ ]:
# ✏️ Self-review for this lesson
l36_review = {
    "windowing_motivation":      None,  # understand why a truncated signal needs windowing (fewer sidelobes)? True/False
    "hann_formula":              None,  # remember Hann: 0.5·(1-cos(2πk/N))? True/False
    "describe_window_impl":      None,  # describe_window implemented and assertions pass? True/False
    "tradeoff_sidelobe_mainlobe": None, # understand the trade-off sidelobes↓ vs main-lobe width↑? True/False
    "whiteboard_passed":         None,  # whiteboard challenge done on paper? True/False
    "ready_for_dft":             None,  # can say that xw = signal * window is the input to L37's DFT? True/False
}

unfilled = [k for k, v in l36_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l36_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L36 fully cleared! On to L37: breaking the windowed xw into frequency components')

---

→ **Next lesson** · [L37 · The brute-force DFT](L37_dft.ipynb)

> Next lesson covers **the brute-force DFT**: `X[k] = Σ x[n] e^{-2πikn/N}`, an O(N²) double loop verified against NumPy.